# WLASL EDA Start

Notebook do szybkiej eksploracji metadanych i spojnosc plikow video.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATASET_DIR = PROJECT_ROOT / 'kaggle_dataset'
META_FILE = DATASET_DIR / 'WLASL_v0.3.json'
VIDEOS_DIR = DATASET_DIR / 'videos'

print('Project root:', PROJECT_ROOT)
print('Metadata exists:', META_FILE.exists())
print('Videos dir exists:', VIDEOS_DIR.exists())

Project root: C:\Users\kacpe\source\repos\szum
Metadata exists: True
Videos dir exists: True


In [2]:
with META_FILE.open('r', encoding='utf-8') as f:
    records = json.load(f)

print('Entries in WLASL_v0.3:', len(records))
print('Keys in first entry:', list(records[0].keys()))
records[0]

Entries in WLASL_v0.3: 2000
Keys in first entry: ['gloss', 'instances']


{'gloss': 'book',
 'instances': [{'bbox': [385, 37, 885, 720],
   'fps': 25,
   'frame_end': -1,
   'frame_start': 1,
   'instance_id': 0,
   'signer_id': 118,
   'source': 'aslbrick',
   'split': 'train',
   'url': 'http://aslbricks.org/New/ASL-Videos/book.mp4',
   'variation_id': 0,
   'video_id': '69241'},
  {'bbox': [190, 25, 489, 370],
   'fps': 25,
   'frame_end': -1,
   'frame_start': 1,
   'instance_id': 1,
   'signer_id': 90,
   'source': 'aslsignbank',
   'split': 'train',
   'url': 'https://aslsignbank.haskins.yale.edu/dictionary/protected_media/glossvideo/ASL/BO/BOOK-418.mp4',
   'variation_id': 0,
   'video_id': '65225'},
  {'bbox': [262, 1, 652, 480],
   'fps': 25,
   'frame_end': -1,
   'frame_start': 1,
   'instance_id': 2,
   'signer_id': 110,
   'source': 'valencia-asl',
   'split': 'train',
   'url': 'https://www.youtube.com/watch?v=0UsjUE-TXns',
   'variation_id': 0,
   'video_id': '68011'},
  {'bbox': [123, 19, 516, 358],
   'fps': 25,
   'frame_end': 60,
   'frame

In [3]:
# Flatten selected metadata for quick statistics
rows = []
for item in records:
    gloss = item.get('gloss')
    for inst in item.get('instances', []):
        rows.append({
            'gloss': gloss,
            'video_id': inst.get('video_id'),
            'split': inst.get('split'),
            'signer_id': inst.get('signer_id'),
        })

df = pd.DataFrame(rows)
df.head()

,gloss,video_id,split,signer_id
0,book,69241,train,118
1,book,65225,train,90
2,book,68011,train,110
3,book,68208,train,113
4,book,68012,train,109


In [4]:
print('Unique glosses:', df['gloss'].nunique())
print('Total instances:', len(df))
print('Split distribution:')
print(df['split'].value_counts(dropna=False))

Unique glosses: 2000
Total instances: 21083
Split distribution:
split
train    14289
val       3916
test      2878
Name: count, dtype: int64


In [5]:
# Check how many videos are present on disk
if VIDEOS_DIR.exists():
    video_files = {p.stem for p in VIDEOS_DIR.glob('*.mp4')}
    expected = set(df['video_id'].dropna().astype(str).unique())
    missing_local = sorted(expected - video_files)
    print('Expected video IDs:', len(expected))
    print('Found local .mp4 files:', len(video_files))
    print('Missing local videos:', len(missing_local))
    missing_local[:20]
else:
    print('videos directory not found')

Expected video IDs: 21083
Found local .mp4 files: 11980
Missing local videos: 9103
